In [1]:
import pandas as pd
import numpy as np
import huggingface 
from models.time_moe_wrapper import Time_MOE_Wrapper
import datetime
import os
import torch
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

print(f"CWD: {os.getcwd()}")

CWD: /home/yinkiat/fyp-kiat/ML_Core/src


/home/yinkiat/.pyenv/versions/time-moe-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading of config
from utils.utils import load_config, config_to_args

config_file_path = "../config/config_tickers_75_7_Channels_with_temporal_tape.yaml"
config_dict = load_config(config_file_path)

place_holders_to_replace = {'version_name': config_dict['regression']['common']['version_name']}
args = config_to_args(config_dict, place_holders_to_replace)

print(f"Current Version Name: {args.regression.common.version_name}")

Current Version Name: tickers_75_7_Channels_with_temporal_tape


### Data Preprocessing into Sliced Sequences ###

In [3]:
### Loading of data and then preprocessing into input needed
from S1_Data_Preprocessing import S1_preprocessing
from S2_Feature_Engineering import S2_feature_engineering
from S3_Model_Training import batched_data_preparation_time_moe, S3_tensor_stages_preparation


full_df = S1_preprocessing(args)

2025-10-27 13:54:29 [INFO] Initiating S1 Data Preprocessing....................
2025-10-27 13:54:29 [INFO] Read in total of 100 tickers...
2025-10-27 13:54:29 [INFO] Preprocessing all alternative data............
/home/yinkiat/fyp-kiat/ML_Core/src/S1_Data_Preprocessing.py:243: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_agg["revenue_growth"] = df_agg.groupby("symbol")["revenue"].pct_change()
2025-10-27 13:54:33 [INFO] Merging with OHLC............
2025-10-27 13:54:40 [INFO] Merged Complete! Total NaN rows: 1902096, Total Rows: 1902096
2025-10-27 13:54:42 [INFO] Saved file to ../data/processed_data/processed_data_tickers_75_7_Channels_with_temporal_tape.parquet


In [4]:
full_df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Ticker',
       'cashGenerationCapitalAllocation', 'growth', 'leverage',
       'profitability', 'score', 'surprise_percent_quarterly',
       'revenue_growth', 'gross_profit', 'operating_profit', 'net_profit',
       'gross_margin', 'operating_margin', 'net_margin', 'current_ratio',
       'quick_ratio', 'roa', 'roe', 'debt_equity', 'dupont_net_margin',
       'dupont_asset_turnover', 'dupont_equity_multiplier', 'dupont_roe',
       'environmentScore', 'governanceScore', 'socialScore', 'totalESGScore',
       'net_change_insider_transactions', 'monthly_share_purchase_ratio',
       'prev_monthly_share_purchase_ratio', 'before_share', 'change', 'share',
       'change_vol', 'transactionPrice', 'social_sentiment_score',
       'totalValue'],
      dtype='object')

In [5]:
# Find duplicated rows by Date and Ticker
dupes = full_df[full_df.duplicated(subset=['Date', 'Ticker'], keep=False)]

# Function to find columns that vary across duplicates
def get_non_duplicate_columns(df, subset_cols):
    non_dup_cols = []
    for col in df.columns:
        if col not in subset_cols:
            # If there's more than one unique value for this column among duplicates
            if df.groupby(subset_cols)[col].nunique().max() > 1:
                non_dup_cols.append(col)
    return non_dup_cols

non_duplicate_columns = get_non_duplicate_columns(dupes, subset_cols=['Date', 'Ticker'])
print(non_duplicate_columns)

[]


In [6]:
# ### Testing the Multi-Frequency ####
# from S3_Model_Training import MultiFrequencyDatasetLoader

# # Initialize the loader
# loader = MultiFrequencyDatasetLoader(
#     args=args,
#     feature_engineered_df=full_df,
#     stage='inference',
#     context_length=24,
#     prediction_length=8,
#     stride=1,
#     batch_size=32,
#     shuffle=True
# )
# # Iterate through batches
# for i, batch in enumerate(loader):
#     print(f"\n===== Batch {i} =====, Keys: {batch.keys()}")
    
#     # Price data (main sequence)
#     input_ids = batch['input_ids']      # [batch, window_size, num_price_features]
#     labels = batch['labels']            # [batch, window_size, num_price_features]
#     loss_masks = batch['loss_masks']    # [batch, window_size]
    
#     print(f"input_ids:   {input_ids.shape}")
#     print(f"labels:      {labels.shape}")
#     print(f"loss_masks:  {loss_masks.shape}")

#     # Multi-frequency channels (Date and Ticker already removed)
#     channel_1 = batch['channel_1']      # [batch, window_size, 128, num_daily_features]
#     channel_2 = batch['channel_2']      # [batch, window_size, 52, num_weekly_features]

#     print(f"channel_1:   {channel_1.shape}")
#     print(f"channel_2:   {channel_2.shape}")

#     # Metadata
#     tickers = batch['ticker']           # List of ticker symbols
#     print(f"tickers:     {len(tickers)} symbols")
#     print(f"example tickers: {tickers[:3]}")  # print first 3 for preview

#     # If you want to print dtypes or devices too:
#     print(f"dtypes: input_ids={input_ids.dtype}, labels={labels.dtype}")
    
    
#     print(f'prediction seq shape: {batch['prediction_sequence'].shape}')
    
#     # Optionally break after the first batch if you just want to inspect
#     break


### S2 Feature Engineering Function ###

In [7]:
feature_engineered_df = S2_feature_engineering(args, full_df)   

2025-10-27 13:54:42 [INFO] Initiating S2 Feature Engineering with 196275 rows currently...............
2025-10-27 13:54:42 [INFO] Saved feature engineered dataframe to ../data/processed_data/feature_engineered_data_tickers_75_7_Channels_with_temporal_tape.parquet
2025-10-27 13:54:42 [INFO] Completed S2 Feature Engineering...............


### S3 Tensor Stages Preparation and saving as Pickle ###